In [3]:
# ============================================================
# INSPECT NEW DATASET BEFORE MERGING
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
PROJECT_DIR = "/content/drive/MyDrive/thesis_movie_project"
RAW_DIR = f"{PROJECT_DIR}/raw"
PROCESSED_DIR = f"{PROJECT_DIR}/processed"

MASTER_PATH = f"{PROCESSED_DIR}/thesis_master_base.csv"

known_old_files = {
    "movies_metadata.csv",
    "keywords.csv",
    "credits.csv",
    "links.csv",
    "ratings_small.csv"
}

# ------------------------------------------------------------
# 1. LIST ALL CSV FILES IN RAW FOLDER
# ------------------------------------------------------------
print("1. CSV files in RAW_DIR:\n")
all_csvs = sorted([f for f in os.listdir(RAW_DIR) if f.lower().endswith(".csv")])

for f in all_csvs:
    marker = " (new candidate)" if f not in known_old_files else ""
    print("-", f, marker)

new_candidates = [f for f in all_csvs if f not in known_old_files]

print("\nDetected new candidates:", new_candidates)

if len(new_candidates) == 0:
    raise FileNotFoundError("No new CSV dataset detected in RAW_DIR.")
elif len(new_candidates) > 1:
    print("\nMore than one new CSV detected.")
    print("The code will inspect the first one automatically, but you may want to set it manually.")

NEW_FILE = os.path.join(RAW_DIR, new_candidates[0])
print("\nUsing NEW_FILE =", NEW_FILE)

# ------------------------------------------------------------
# 2. LOAD NEW DATASET
# ------------------------------------------------------------
print("\n2. Loading new dataset ...")
new_df = pd.read_csv(NEW_FILE, low_memory=False)

print("New dataset shape:", new_df.shape)
print("\nColumns:\n")
for c in new_df.columns:
    print("-", c)

print("\nPreview:")
display(new_df.head(10))

# ------------------------------------------------------------
# 3. BASIC MISSINGNESS
# ------------------------------------------------------------
print("\n3. Missing values (%) in new dataset:")
missing_pct = (new_df.isna().mean() * 100).round(2).sort_values(ascending=False)
print(missing_pct)

# ------------------------------------------------------------
# 4. CHECK LIKELY KEY COLUMNS
# ------------------------------------------------------------
print("\n4. Key column diagnostics:")

candidate_keys = ["id", "movieId", "imdb_id", "imdbId", "tmdbId", "title", "original_title", "release_date"]
for col in candidate_keys:
    if col in new_df.columns:
        print(f"\nColumn: {col}")
        print("  Non-null:", new_df[col].notna().sum())
        print("  Unique:", new_df[col].nunique(dropna=True))
        print("  Sample:", new_df[col].dropna().astype(str).head(5).tolist())

# ------------------------------------------------------------
# 5. TRY TO PARSE RELEASE DATE / YEAR
# ------------------------------------------------------------
print("\n5. Release date and year diagnostics:")

release_col = None
for c in ["release_date", "Release_Date", "date", "year", "releaseYear"]:
    if c in new_df.columns:
        release_col = c
        break

if release_col is not None:
    print("Using release column:", release_col)

    if "year" in release_col.lower() and release_col.lower() != "release_date":
        new_df["release_year_tmp"] = pd.to_numeric(new_df[release_col], errors="coerce")
        new_df["release_date_tmp"] = pd.NaT
    else:
        new_df["release_date_tmp"] = pd.to_datetime(new_df[release_col], errors="coerce")
        new_df["release_year_tmp"] = new_df["release_date_tmp"].dt.year

    print("Release year summary:")
    print(new_df["release_year_tmp"].describe())

    print("\nMovies by release year (tail 20 years):")
    year_counts = new_df["release_year_tmp"].value_counts(dropna=True).sort_index()
    print(year_counts.tail(20))

    current_date = pd.Timestamp.today().normalize()
    if "release_date_tmp" in new_df.columns:
        unreleased_count = (new_df["release_date_tmp"] > current_date).sum()
        print(f"\nUnreleased rows detected (release_date > today): {unreleased_count}")
else:
    print("No obvious release date/year column found.")

# ------------------------------------------------------------
# 6. CHECK FINANCE COLUMNS
# ------------------------------------------------------------
print("\n6. Finance diagnostics:")

for col in ["budget", "revenue", "Budget", "Revenue"]:
    if col in new_df.columns:
        numeric_col = pd.to_numeric(new_df[col], errors="coerce")
        print(f"\nColumn: {col}")
        print("  Non-null:", numeric_col.notna().sum())
        print("  > 0:", (numeric_col > 0).sum())
        print("  == 0:", (numeric_col == 0).sum())
        print("  Missing:", numeric_col.isna().sum())
        print("  Describe:")
        print(numeric_col.describe())

# ------------------------------------------------------------
# 7. LOAD CURRENT MASTER AND COMPARE SCHEMA
# ------------------------------------------------------------
print("\n7. Comparing with thesis_master.csv ...")

master = pd.read_csv(MASTER_PATH, low_memory=False)

print("Master shape:", master.shape)

master_cols = set(master.columns)
new_cols = set(new_df.columns)

common_cols = sorted(master_cols.intersection(new_cols))
master_only = sorted(master_cols - new_cols)
new_only = sorted(new_cols - master_cols)

print("\nCommon columns:")
print(common_cols)

print("\nColumns only in master:")
print(master_only)

print("\nColumns only in new dataset:")
print(new_only)

# ------------------------------------------------------------
# 8. TRY TO ESTIMATE OVERLAP USING COMMON KEYS
# ------------------------------------------------------------
print("\n8. Overlap diagnostics ...")

# Try using imdb_id if available
if "imdb_id" in master.columns and "imdb_id" in new_df.columns:
    master_imdb = set(master["imdb_id"].dropna().astype(str))
    new_imdb = set(new_df["imdb_id"].dropna().astype(str))
    overlap_imdb = len(master_imdb.intersection(new_imdb))
    print("IMDb ID overlap:", overlap_imdb)

# Try using id if available
if "id" in master.columns and "id" in new_df.columns:
    master_id = set(pd.to_numeric(master["id"], errors="coerce").dropna().astype(int))
    new_id = set(pd.to_numeric(new_df["id"], errors="coerce").dropna().astype(int))
    overlap_id = len(master_id.intersection(new_id))
    print("TMDB ID overlap:", overlap_id)

# Try title + year if possible
if "title" in master.columns and "title" in new_df.columns and "release_year_tmp" in new_df.columns:
    tmp_master = master.copy()
    tmp_master["title_key"] = tmp_master["title"].fillna("").astype(str).str.strip().str.lower()
    tmp_new = new_df.copy()
    tmp_new["title_key"] = tmp_new["title"].fillna("").astype(str).str.strip().str.lower()

    if "release_year" in tmp_master.columns:
        master_pairs = set(zip(tmp_master["title_key"], tmp_master["release_year"]))
        new_pairs = set(zip(tmp_new["title_key"], tmp_new["release_year_tmp"]))
        overlap_pairs = len(master_pairs.intersection(new_pairs))
        print("Title + year overlap:", overlap_pairs)

# ------------------------------------------------------------
# 9. SUGGESTION SUMMARY
# ------------------------------------------------------------
print("\n9. Preliminary recommendation:")

print("- If the new dataset has recent released movies (2019-2026), it is worth merging.")
print("- If it contains many unreleased titles, those should be excluded from the main thesis dataset.")
print("- If revenue coverage is much better, finance analysis improves.")
print("- If budget is mostly zeros, keep treating budget <= 0 as missing.")
print("- Final merge priority should likely be: id -> imdb_id -> title+release_year")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. CSV files in RAW_DIR:

- IMDB TMDB Movie Metadata Big Dataset (1M).csv  (new candidate)
- credits.csv 
- keywords.csv 
- links.csv 
- movies_metadata.csv 
- ratings_small.csv 

Detected new candidates: ['IMDB TMDB Movie Metadata Big Dataset (1M).csv']

Using NEW_FILE = /content/drive/MyDrive/thesis_movie_project/raw/IMDB TMDB Movie Metadata Big Dataset (1M).csv

2. Loading new dataset ...
New dataset shape: (1072255, 42)

Columns:

- id
- title
- vote_average
- vote_count
- status
- release_date
- revenue
- runtime
- adult
- backdrop_path
- budget
- homepage
- imdb_id
- original_language
- original_title
- overview
- popularity
- poster_path
- tagline
- production_companies
- production_countries
- spoken_languages
- keywords
- release_year
- Director
- AverageRating
- Poster_Link
- Certificate
- IMDB_Rating
- Meta_score
- Star1
- Star2
- Star3
- Star4
- W

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,Star3,Star4,Writer,Director_of_Photography,Producers,Music_Composer,genres_list,Cast_list,overview_sentiment,all_combined_keywords
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,Elliot Page,Ken Watanabe,Christopher Nolan,Wally Pfister,"Thomas Tull, Christopher Nolan, Chris Brigham,...",Hans Zimmer,"['Action', 'Science Fiction', 'Adventure']","['Tim Kelleher', 'Silvie Laguna', 'Natasha Bea...",-0.011111,"['s', 'philosophy', 'skilled', 'kidnapping', '..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,Jessica Chastain,Mackenzie Foy,Jonathan Nolan,Hoyte van Hoytema,"Jake Myers, Emma Thomas, Jordan Goldberg, Thom...",Hans Zimmer,"['Adventure', 'Drama', 'Science Fiction']","['Jeff Hephner', 'William Devane', 'Elyes Gabe...",0.045455,"['thoughtful', 'use', 'scientist', 'quantum me..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,Aaron Eckhart,Michael Caine,Jonathan Nolan,Wally Pfister,"Kevin De La Noy, Thomas Tull, Christopher Nola...","Hans Zimmer, James Newton Howard","['Drama', 'Action', 'Crime', 'Thriller']","['Tommy Lister Jr.', 'Edison Chen', 'Beatrice ...",0.025000,"['reign', 'harvey', 'proves', 'partnership', '..."
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,Sigourney Weaver,Michelle Rodriguez,James Cameron,Mauro Fiore,"Jon Landau, Colin Wilson, Laeta Kalogridis, Ja...",James Horner,"['Action', 'Adventure', 'Fantasy', 'Science Fi...","['Carvon Futrell', 'Joel David Moore', 'Jon Cu...",0.041667,"['space war', 'civilization', 'society', 'mari..."
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,Scarlett Johansson,Jeremy Renner,Sydney Newman,NaN,NaN,NaN,"['Science Fiction', 'Action', 'Adventure']","['Haneyuri', 'Nako Mizusawa', 'Marin', 'Rikako...",0.025000,"['s', 'world', 'begins', 'e', 'global safety',..."
5,293660,Deadpool,7.606,28894,Released,2016-02-09,783100000,108,False,/en971MEXui9diirXlogOrPKmsEn.jpg,...,T.J. Miller,Ed Skrein,Joss Whedon,NaN,NaN,NaN,"['Action', 'Adventure', 'Comedy']","['Tulsi Lahiri', 'Ahindra Choudhury', 'Devidas...",-0.030293,"['wade', 'experiment', 'subjected', 'wade wils..."
6,299536,Avengers: Infinity War,8.255,27713,Released,2018-04-25,2052415039,149,False,/mDfJG3LC3Dqb67AZ52x3Z0jU0uB.jpg,...,Chris Hemsworth,Mark Ruffalo,Rhett Reese,NaN,Joan Ginard,NaN,"['Adventure', 'Action', 'Science Fiction']","['Gerard Villegas', 'Sebastián Haro', 'Rosana ...",0.087662,"['world', 'protect', 'use', 'thanos', 'infinit..."
7,550,Fight Club,8.438,27238,Released,1999-10-15,100853753,139,False,/hZkgoQYus5vegHoetLkCJzb17zJ.jpg,...,Meat Loaf,Zach Grenier,Christopher Markus,NaN,NaN,NaN,['Drama'],"['Augusto Mastripietri', 'Amleto Novelli', 'An...",-0.215909,"['fight clubs', 'way', 'control', 'fight', 'fo..."
8,118340,Guardians of the Galaxy,7.906,26638,Released,2014-07-30,772776600,121,False,/uLtVbjvS1O7gXL8lUOwsFOH4man.jpg,...,Bradley Cooper,Zoe Saldana,Chuck Palahniuk,Roger Pratt,"Susan Ekins, Jerry Weintraub",Joel McNeely,"['Action', 'Science Fiction', 'Adventure']","['David Webber', 'Uma Thurman', 'Fiona Shaw', ...",0.400000,"['orb', 'ronan', 'space', 'light', 'prime targ..."
9,680,Pulp Fiction,8.488,25893,Released,1994-09-10,213900000,154,False,/suaEOtk1N1sgg2MTM7oZd2cfVp3.jpg,...,Samuel L. Jackson,Bruce Willis,James Gunn,Seamus McGarvey,"Louis D'Esposito, Jon Favreau, Avi Arad, Kevin...",Alan Silvestri,"['Thriller', 'Crime']","['Israel Hall', 'Brent McGee', 'Powers Boothe'...",0.166667,"['s', 'heirloom', 'massage', 'moll', 'neo-noir..."



3. Missing values (%) in new dataset:
Meta_score                 99.78
Star2                      99.74
Star3                      99.74
Poster_Link                99.74
Star4                      99.74
AverageRating              98.90
Certificate                97.22
IMDB_Rating                97.17
Star1                      97.17
homepage                   89.23
tagline                    85.75
Music_Composer             85.50
backdrop_path              72.69
Director_of_Photography    68.58
Producers                  60.34
production_companies       53.69
imdb_id                    45.04
production_countries       42.56
spoken_languages           41.06
Writer                     39.70
poster_path                29.64
overview                   18.86
release_date               14.04
release_year               14.04
id                          0.00
title                       0.00
adult                       0.00
runtime                     0.00
revenue                     0.00
stat